# 04 — Weekly Matchup & Monte Carlo Dashboard

This notebook provides an interactive visual breakdown of your weekly head-to-head matchup.  It pulls live Sleeper league data, optimizes starting lineups with DvP adjustments, runs a Monte Carlo simulation to estimate win probability, and plots positional advantages.

**Workflow**
1. League & Matchup Sync
2. Optimal Lineup vs. Opponent Starters
3. Monte Carlo Outcome Distribution
4. Positional Advantage Delta Chart
5. Sit/Start Toss-Up Explorer

In [1]:
import os, sys

# Auto-resolve project root (handles running from notebooks/ subdirectory)
if not os.path.exists('data/projections.csv'):
    os.chdir('..')
if '.' not in sys.path:
    sys.path.insert(0, '.')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.league import (
    fetch_league, fetch_league_users, fetch_league_rosters,
    fetch_league_matchups, parse_users_dataframe, parse_rosters_dataframe,
)
from src.sleeper import fetch_sleeper_players, parse_sleeper_catalog
from src.matchups import extract_weekly_matchup_roster, optimize_starting_lineup
from src.simulation import simulate_team_matchup
from src.dvp import calculate_defensive_rankings, adjust_projections_for_matchup
from src.injuries import extract_roster_injury_status
from src.cli import _load_draft_board, _load_weekly_stats

POS_COLORS = {'QB': '#4C78A8', 'RB': '#54A24B', 'WR': '#E45756', 'TE': '#F58518'}
TEAM_A_COLOR = '#2980B9'
TEAM_B_COLOR = '#E74C3C'

plt.style.use('seaborn-v0_8-whitegrid')

LEAGUE_ID = '1386423085304938496'
ROSTER_ID = 14
WEEK = 1

## 1 — League & Matchup Sync

Connect to the Sleeper API, resolve the league manager display names, roster ownership, and identify this week’s matchup pairing.

In [2]:
league = fetch_league(LEAGUE_ID)
users_raw = fetch_league_users(LEAGUE_ID)
users_df = parse_users_dataframe(users_raw)
user_lookup = dict(zip(users_df['user_id'], users_df['display_name']))

rosters_raw = fetch_league_rosters(LEAGUE_ID)
rosters_df = parse_rosters_dataframe(rosters_raw)

matchups_raw = fetch_league_matchups(LEAGUE_ID, WEEK)

our_matchup_data = None
opponent_matchup_data = None
for m in matchups_raw:
    if m.get('roster_id') == ROSTER_ID:
        our_matchup_data = m
        for opp in matchups_raw:
            if (opp.get('matchup_id') == m.get('matchup_id')
                    and opp.get('roster_id') != ROSTER_ID):
                opponent_matchup_data = opp
                break
        break

our_owner = 'You'
for r in rosters_raw:
    if r.get('roster_id') == ROSTER_ID:
        our_owner = user_lookup.get(r.get('owner_id', ''), 'You')
        break

opp_id = opponent_matchup_data.get('roster_id', '?') if opponent_matchup_data else '?'
opp_owner = 'Opponent'
for r in rosters_raw:
    if r.get('roster_id') == opp_id:
        opp_owner = user_lookup.get(r.get('owner_id', ''), 'Opponent')
        break

print(f'Week {WEEK} Matchup')
print(f'  You:   Roster {ROSTER_ID} ({our_owner})')
print(f'  vs.')
print(f'  Them:  Roster {opp_id} ({opp_owner})')

Week 1 Matchup
  You:   Roster 10 (eltor0day)
  vs.
  Them:  Roster 2 (LessThanTacos)


## 2 — Optimal Lineup vs. Opponent Starters

Load the VORP draft board, compute weekly projections (÷17), apply DvP matchup multipliers, and optimize starting lineups for both teams.

In [6]:
board = _load_draft_board()
catalog = parse_sleeper_catalog(fetch_sleeper_players())

proj = board[['player_id', 'player_name', 'position_proj', 'proj_points', 'team']].copy()
proj.rename(columns={'position_proj': 'position'}, inplace=True)
proj['proj_points'] = proj['proj_points'] / 17.0

weekly_stats = _load_weekly_stats()
dvp_ranks = calculate_defensive_rankings(weekly_stats)
week_stats = weekly_stats[weekly_stats['week'] == WEEK]
schedule = week_stats[['team', 'opponent_team']].drop_duplicates()
adj_proj = adjust_projections_for_matchup(proj, schedule, dvp_ranks)

player_pool = board[['player_id', 'player_name', 'position_proj']].copy()
player_pool.rename(columns={'position_proj': 'position'}, inplace=True)

our_m = extract_weekly_matchup_roster(ROSTER_ID, matchups_raw, player_pool)
our_pids = our_m['player_id'].tolist()
opt_proj = adj_proj[['player_id', 'position', 'player_name', 'adjusted_proj_points']].copy()
opt_proj.rename(columns={'adjusted_proj_points': 'proj_points'}, inplace=True)
opt_our = optimize_starting_lineup(our_pids, opt_proj)

opp_roster_id = int(opponent_matchup_data.get('roster_id', 0)) if opponent_matchup_data else 0
opp_m = extract_weekly_matchup_roster(opp_roster_id, matchups_raw, player_pool)
opp_pids = opp_m['player_id'].tolist()
opt_opp = optimize_starting_lineup(opp_pids, opt_proj)

all_pids = list(set(our_pids + opp_pids))
injury_df = extract_roster_injury_status(all_pids, catalog)
injury_lookup = dict(zip(injury_df['player_id'], injury_df['injury_tag'])) if not injury_df.empty else {}
name_lookup = dict(zip(board['player_id'], board['player_name']))
pos_lookup = dict(zip(board['player_id'], board['position_proj']))

dvp_lookup = {}
for _, r in adj_proj.iterrows():
    dvp_lookup[r['player_id']] = {
        'opponent': r.get('opponent', ''),
        'defense_rank': r.get('defense_rank', 16),
        'multiplier': r.get('multiplier', 1.0),
    }

def _fmt_lineup(opt, dvp_lk, adj, name_lk, pos_lk, inj_lk):
    rows = []
    for slot in sorted(opt['lineup']):
        pid = opt['lineup'][slot]
        dvp = dvp_lk.get(pid, {})
        m = adj.loc[adj['player_id'] == pid, 'adjusted_proj_points']
        pts = round(float(m.iloc[0]), 1) if not m.empty else 0.0
        rows.append({
            'Slot': slot,
            'Player': f"{name_lk.get(pid, pid)}{inj_lk.get(pid, '')}",
            'Pos': pos_lk.get(pid, '?'),
            'PPG': pts,
            'DvP': dvp.get('defense_rank', 16),
            'Mult': f"{dvp.get('multiplier', 1.0):.2f}",
            'Opp': dvp.get('opponent', ''),
        })
    return pd.DataFrame(rows)

your_df = _fmt_lineup(opt_our, dvp_lookup, adj_proj, name_lookup, pos_lookup, injury_lookup)
opp_df = _fmt_lineup(opt_opp, dvp_lookup, adj_proj, name_lookup, pos_lookup, injury_lookup)

print(f'Your projected total: {opt_our["total_proj_points"]:.1f} pts')
print(f'Opponent projected:  {opt_opp["total_proj_points"]:.1f} pts')
print()
print('=== YOUR STARTERS ===')
your_df

FileNotFoundError: [Errno 2] No such file or directory: 'data/projections.csv'

In [4]:
print('=== OPPONENT STARTERS ===')
opp_df

=== OPPONENT STARTERS ===


NameError: name 'opp_df' is not defined

## 3 — Monte Carlo Outcome Distribution

Run 10,000 simulated matchups and plot the overlaid score distributions for both teams.  The shaded regions show the 10th–90th percentile bands, with the median marked by dashed lines.

In [ ]:
from src.simulation import simulate_player_weekly_distribution

def _build_starter_df(lineup_dict):
    rows = []
    for pid in lineup_dict.values():
        matches = adj_proj[adj_proj['player_id'] == pid]
        if matches.empty:
            continue
        rows.append({
            'player_id': pid,
            'position': pos_lookup.get(pid, '?'),
            'proj_points': float(matches['adjusted_proj_points'].iloc[0]),
        })
    return pd.DataFrame(rows)

our_starters_df = _build_starter_df(opt_our['lineup'])
opp_starters_df = _build_starter_df(opt_opp['lineup'])

sim = simulate_team_matchup(our_starters_df, opp_starters_df, iterations=10000, random_seed=42)

def _sim_total(starters_df, n_iter=10000, seed=42):
    total = np.zeros(n_iter)
    for _, row in starters_df.iterrows():
        total += simulate_player_weekly_distribution(
            row['proj_points'], iterations=n_iter, random_seed=seed
        )
    return total

our_sims = _sim_total(our_starters_df, seed=42)
opp_sims = _sim_total(opp_starters_df, seed=123)

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(
    min(our_sims.min(), opp_sims.min()) - 5,
    max(our_sims.max(), opp_sims.max()) + 5,
    60,
)
ax.hist(our_sims, bins=bins, alpha=0.45, color=TEAM_A_COLOR, density=True,
        label=f'You (Roster {ROSTER_ID})')
ax.hist(opp_sims, bins=bins, alpha=0.45, color=TEAM_B_COLOR, density=True,
        label=f'Opponent (Roster {opp_roster_id})')

for sims, color, label in [
    (our_sims, TEAM_A_COLOR, 'You'),
    (opp_sims, TEAM_B_COLOR, 'Opponent'),
]:
    med = np.median(sims)
    p10 = np.percentile(sims, 10)
    p90 = np.percentile(sims, 90)
    ax.axvline(med, color=color, linestyle='--', linewidth=2,
               label=f'{label} Median: {med:.1f}')
    ax.axvline(p10, color=color, linestyle=':', linewidth=1, alpha=0.7)
    ax.axvline(p90, color=color, linestyle=':', linewidth=1, alpha=0.7)
    ylim = ax.get_ylim()
    ax.fill_betweenx([0, ylim[1] if ylim[1] > 0 else 0.01], p10, p90,
                      color=color, alpha=0.08)

win_pct = sim['win_prob_a'] * 100
ax.set_title(
    f'Monte Carlo Score Distribution ({sim["iterations"]:,} sims)\n'
    f'Win Probability: You {win_pct:.1f}% | Opponent {100 - win_pct:.1f}%',
    fontsize=13, fontweight='bold',
)
ax.set_xlabel('Simulated Weekly Points')
ax.set_ylabel('Density')
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
os.makedirs('reports', exist_ok=True)
fig.savefig('reports/week_matchup_mc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSimulation Summary:')
print(f'  You:      Median {sim["median_a"]:.1f} | Floor {sim["floor_a"]:.1f} | Ceiling {sim["ceiling_a"]:.1f}')
print(f'  Opponent: Median {sim["median_b"]:.1f} | Floor {sim["floor_b"]:.1f} | Ceiling {sim["ceiling_b"]:.1f}')
print(f'  Win Prob: {win_pct:.1f}% | Avg Margin: {sim["avg_margin"]:+.1f}')

## 4 — Positional Advantage Delta Chart

For each matching starter slot, plot the point differential (Your PPG − Opponent PPG).  Green bars indicate a positional advantage; red bars indicate a deficit.

In [ ]:
our_by_pos = {}
for pid in opt_our['lineup'].values():
    matches = adj_proj[adj_proj['player_id'] == pid]
    if matches.empty:
        continue
    pos = pos_lookup.get(pid, '?')
    our_by_pos[pos] = our_by_pos.get(pos, 0.0) + float(matches['adjusted_proj_points'].iloc[0])

opp_by_pos = {}
for pid in opt_opp['lineup'].values():
    matches = adj_proj[adj_proj['player_id'] == pid]
    if matches.empty:
        continue
    pos = pos_lookup.get(pid, '?')
    opp_by_pos[pos] = opp_by_pos.get(pos, 0.0) + float(matches['adjusted_proj_points'].iloc[0])

all_positions = sorted(set(list(our_by_pos.keys()) + list(opp_by_pos.keys())))
delta_rows = []
for pos in all_positions:
    a = our_by_pos.get(pos, 0.0)
    b = opp_by_pos.get(pos, 0.0)
    delta_rows.append({'Position': pos, 'Team A': round(a, 1), 'Team B': round(b, 1), 'Delta': round(a - b, 1)})

delta_df = pd.DataFrame(delta_rows).sort_values('Delta', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = np.arange(len(delta_df))
colors = ['#27AE60' if d >= 0 else '#E74C3C' for d in delta_df['Delta']]

ax.barh(y_pos, delta_df['Delta'], color=colors, edgecolor='white', height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(delta_df['Position'], fontsize=11)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Point Differential (You - Opponent)', fontsize=11)
ax.set_title('Positional Advantage Delta', fontsize=13, fontweight='bold')

for i, row in enumerate(delta_df.itertuples()):
    sign = '+' if row.Delta >= 0 else ''
    offset = 0.3 if row.Delta >= 0 else -0.3
    ha = 'left' if row.Delta >= 0 else 'right'
    ax.text(row.Delta + offset, i, f'{sign}{row.Delta:.1f}',
            va='center', ha=ha, fontsize=9, fontweight='bold')

fig.tight_layout()
fig.savefig('reports/week_matchup_delta.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPositional Breakdown:')
print(delta_df.to_string(index=False))

## 5 — Sit/Start Toss-Up Explorer

Identify bench vs. starter candidates within ±3.0 projected points.  These are the "coin-flip" decisions where the matchup analysis can tip the scale.

In [ ]:
from src.matchups import compare_sit_start, _TOSSUP_TOLERANCE

roster_projs = adj_proj[adj_proj['player_id'].isin(our_pids)][
    ['player_id', 'player_name', 'position', 'adjusted_proj_points']
].copy()
roster_projs.rename(columns={'adjusted_proj_points': 'proj_points'}, inplace=True)

starter_set = set(opt_our['lineup'].values())
bench_pids = [p for p in our_pids if p not in starter_set]
starter_pids = [p for p in our_pids if p in starter_set]

tossup_rows = []
for s_pid in starter_pids:
    s_match = roster_projs[roster_projs['player_id'] == s_pid]
    if s_match.empty:
        continue
    s_pts = float(s_match['proj_points'].iloc[0])
    s_name = name_lookup.get(s_pid, s_pid)
    s_pos = pos_lookup.get(s_pid, '?')

    for b_pid in bench_pids:
        b_match = roster_projs[roster_projs['player_id'] == b_pid]
        if b_match.empty:
            continue
        b_pts = float(b_match['proj_points'].iloc[0])
        b_name = name_lookup.get(b_pid, b_pid)
        b_pos = pos_lookup.get(b_pid, '?')

        if abs(s_pts - b_pts) <= _TOSSUP_TOLERANCE and s_pos == b_pos:
            result = compare_sit_start(s_pid, b_pid, roster_projs)
            tossup_rows.append({
                'Starter': f'{s_name} ({s_pos})',
                'Starter PPG': round(s_pts, 1),
                'Bench': f'{b_name} ({b_pos})',
                'Bench PPG': round(b_pts, 1),
                'Delta': round(s_pts - b_pts, 1),
                'Recommendation': result['recommendation'],
            })

if tossup_rows:
    tossup_df = pd.DataFrame(tossup_rows).sort_values('Delta', key=abs)
    print(f'Found {len(tossup_df)} toss-up candidate(s) within {_TOSSUP_TOLERANCE} pts:')
    tossup_df
else:
    print(f'No toss-up candidates found within +/-{_TOSSUP_TOLERANCE} pts of each other.')